# 🧠 DAY 8: LLM FUNDAMENTALS, TOKENISATION & INFERENCE HYPERPARAMETERS
### End-to-End Notebook: From Attention Mechanics to Temperature & Top-P Divergence

This notebook covers:
1. **Environment Setup & Gemini API Ingestion**
2. **From N-Grams to Transformers & Tokenisation Mechanics** (Subwords, BPE)
3. **Self-Attention Intuition & Matrix Math** ($Q, K, V$ scaled dot-product)
4. **The 3-Stage Training Pipeline** (Pre-training $\rightarrow$ SFT $\rightarrow$ RLHF/DPO)
5. **Inference Hyperparameters Math** (Temperature, Nucleus Sampling / Top-P, Top-K)
6. **Live Demo 1: Temperature Divergence Experiment**
7. **Live Demo 2: Pre-flight Token Counting & Financial Cost Calculator**
8. **Hallucinations, Safety & Foundation Model Landscape**
9. **Section 8: Student Graded Lab Assignment — Systematic Hyperparameter Divergence on Technical Debt**

In [ ]:
# ==============================================================================
# CELL 0: ENVIRONMENT SETUP & GOOGLE GEMINI API AUTHENTICATION
# Run this cell first. Installs the official Google GenAI SDK and configures auth.
# ==============================================================================
!pip install -q -U google-genai tiktoken tabulate python-dotenv numpy pandas matplotlib seaborn

import os
import time
import math
import getpass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate
from dotenv import load_dotenv

# Modern Google GenAI SDK
from google import genai
from google.genai import types

# Plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["font.size"] = 10

# ------------------------------------------------------------------------------
# Secure Gemini API Key Ingestion
# ------------------------------------------------------------------------------
load_dotenv()

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("🔑 Enter your Google Gemini API Key: ")
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

MODEL_NAME = os.environ.get('GEMINI_MODEL', 'gemini-3-flash-preview')

# Initialize the Gemini Client
client = genai.Client(api_key=GEMINI_API_KEY)
print(f"✅ Google Gemini Client initialized successfully! (Model: {MODEL_NAME})")

# ==============================================================================
# SECTION 1: FROM N-GRAMS TO TRANSFORMERS & TOKENISATION MECHANICS
# ==============================================================================
### The Architectural Evolution:
1. **N-Grams (Statistical Probability)**: Limited context window $N-1$, suffers from combinatorial explosion.
2. **RNNs / LSTMs**: Sequential processing bottleneck ($h_t = f(Wh_{t-1} + Ux_t)$), vanishing gradients, non-parallelizable on GPUs.
3. **Transformers (Vaswani et al., 2017)**: Self-attention calculates all pairwise token relationships in parallel across entire documents.

In [ ]:
# Visualizing Subword Token Splitting Mechanics
try:
    import tiktoken
    tokenizer = tiktoken.get_encoding("cl100k_base")

    def inspect_tokenisation(text_sample: str):
        token_ids = tokenizer.encode(text_sample)
        token_strings = [tokenizer.decode([t_id]) for t_id in token_ids]

        print(f"Input Text: '{text_sample}'")
        print(f"Total Characters: {len(text_sample)} | Total Words: {len(text_sample.split())} | Total Tokens: {len(token_ids)}")
        print("Tokenized Chunks:")
        for idx, (t_id, t_str) in enumerate(zip(token_ids, token_strings)):
            print(f"  [Chunk {idx+1}]: Token ID {t_id:6d} -> '{t_str}'")
        print("-" * 60)

    print("=== TOKENISATION INSPECTION EXAMPLES ===\n")
    inspect_tokenisation("strawberry")
    inspect_tokenisation("Unbelievable, anti-constitutional transformations occurred in 2026.")
    inspect_tokenisation("def calculate_fibonacci(n: int) -> int:")
except ImportError:
    print("Install tiktoken (!pip install tiktoken) to inspect BPE token chunks.")

# ==============================================================================
# SECTION 2: TRANSFORMER ARCHITECTURE & SELF-ATTENTION (Q, K, V)
# ==============================================================================
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$
- **Query (Q)**: What a given token searches for.
- **Key (K)**: What a given token advertises.
- **Value (V)**: The contextual information transferred.

In [ ]:
# Numerical Demonstration of Scaled Dot-Product Attention in Pure NumPy
np.random.seed(42)
seq_len = 4
d_k = 8  # Dimension of query/key vectors

# Simulated token representations: ["The", "bank", "robbery", "failed"]
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

# 1. Compute Raw Attention Scores
scores = np.dot(Q, K.T) / np.sqrt(d_k)

# 2. Apply Softmax along rows
attention_weights = np.exp(scores) / np.sum(np.exp(scores), axis=-1, keepdims=True)

# 3. Compute Contextual Output
context_output = np.dot(attention_weights, V)

tokens = ["The", "bank", "robbery", "failed"]
plt.figure(figsize=(6, 4.5))
sns.heatmap(attention_weights, xticklabels=tokens, yticklabels=tokens, annot=True, fmt=".2f", cmap="Blues")
plt.title("Self-Attention Alignment Matrix (Softmax Scaled)", fontweight="bold")
plt.xlabel("Key Tokens (K)")
plt.ylabel("Query Tokens (Q)")
plt.tight_layout()
plt.show()

# ==============================================================================
# SECTION 3: THE 3-STAGE TRAINING PIPELINE
# ==============================================================================
1. **Self-Supervised Pre-Training**: Unsupervised next-token prediction across trillions of web/code tokens.
2. **Supervised Fine-Tuning (SFT / Instruction Tuning)**: Demonstrations of prompt $\rightarrow$ high-quality answer format.
3. **Reinforcement Learning from Human Feedback (RLHF / DPO)**: Aligning safety, truthfulness, and helpfulness via pairwise reward ranking.

# ==============================================================================
# SECTION 4: INFERENCE HYPERPARAMETERS MATH (TEMPERATURE, TOP-P, TOP-K)
# ==============================================================================
$$P(w_i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

In [ ]:
# Visualizing the Mathematical Impact of Temperature on Softmax Logits
candidate_words = ["algorithm", "pipeline", "banana", "galaxy", "quantum"]
raw_logits = np.array([4.2, 3.5, 0.5, 1.2, 2.8])

temperatures = [0.2, 0.7, 1.5]
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8), sharey=True)

for ax, t in zip(axes, temperatures):
    scaled_exp = np.exp(raw_logits / t)
    probs = scaled_exp / np.sum(scaled_exp)

    bars = ax.bar(candidate_words, probs, color="#1e40af", width=0.6)
    ax.set_title(f"Temperature T = {t}", fontweight="bold")
    ax.set_ylabel("Sampling Probability")
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=25)

    for bar in bars:
        h = bar.get_height()
        if h > 0.05:
            ax.text(bar.get_x() + bar.get_width()/2., h + 0.02, f"{h*100:.1f}%", ha='center', fontsize=9, fontweight='bold')

plt.suptitle("Softmax Probability Distribution Flattening under Rising Temperature", fontsize=12, y=1.05, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# SECTION 5: LIVE DEMO 1 — TEMPERATURE DIVERGENCE EXPERIMENT
# ==============================================================================
prompt_test = "Write a 2-sentence marketing slogan for an AI-powered coffee maker."
temp_values = [0.0, 0.7, 1.5]
divergence_results = []

print(f"Executing Gemini API Prompt: '{prompt_test}'\n")

for temp in temp_values:
    config = types.GenerateContentConfig(
        temperature=temp,
        max_output_tokens=100,
        top_p=0.95
    )

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_test,
        config=config
    )

    if response.text:
        generated_text = response.text.strip()
    else:
        generated_text = "[NO TEXT GENERATED]"

    divergence_results.append({
        "Temperature": temp,
        "Behavioral Regime": "Deterministic / Greedy" if temp == 0.0 else ("Balanced Standard" if temp == 0.7 else "High Entropy / Creative"),
        "Generated Response": generated_text
    })
    time.sleep(1.0)

display(pd.DataFrame(divergence_results))

In [ ]:
# ==============================================================================
# SECTION 6: LIVE DEMO 2 — TOKEN COUNTING & FINANCIAL COST CALCULATOR
# ==============================================================================
enterprise_input_text = """
CloudPulse Corporation Quarterly Financial Briefing (Q3 FY2026):
Our enterprise subscription gross revenue reached $12.4M representing a 28% YoY expansion.
Operating expenditures aggregated to $3.4M primarily driven by GPU infrastructure and data center lease obligations.
Net profit margin for the fiscal quarter closed at 18.2%, surpassing Wall Street consensus estimates.
"""

token_count_resp = client.models.count_tokens(
    model=MODEL_NAME,
    contents=enterprise_input_text
)
input_tokens = token_count_resp.total_tokens

print("=== 📊 GEMINI TOKEN COUNTING & COST MODELING ===")
print(f"Document Character Length: {len(enterprise_input_text):,} characters")
print(f"Document Word Count:       {len(enterprise_input_text.split()):,} words")
print(f"Official Gemini Tokens:    {input_tokens:,} tokens")

def calculate_gemini_cost(n_input: int, n_output: int, model: str = MODEL_NAME) -> float:
    pricing_matrix = {
        "gemini-3-flash-preview": {"input_per_m": 0.075, "output_per_m": 0.30},
        "gemini-flash-latest":    {"input_per_m": 0.075, "output_per_m": 0.30},
        "gemini-2.5-flash":       {"input_per_m": 0.075, "output_per_m": 0.30},
        "gemini-1.5-pro":         {"input_per_m": 1.25,  "output_per_m": 5.00}
    }
    rates = pricing_matrix.get(model, pricing_matrix["gemini-3-flash-preview"])
    return (n_input / 1_000_000 * rates["input_per_m"]) + (n_output / 1_000_000 * rates["output_per_m"])

sample_output_tokens = 250
est_cost = calculate_gemini_cost(input_tokens, sample_output_tokens, MODEL_NAME)
print(f"Estimated Cost for 1,000 Batched Invocations: ${est_cost * 1000:.4f} USD")

In [ ]:
# ==============================================================================
# SECTION 7: HALLUCINATIONS, SAFETY & FOUNDATION MODEL LANDSCAPE
# ==============================================================================
model_comparison_data = [
    ["Google Gemini 2.5 Flash", "Google", "Proprietary / API", "1M Tokens", "Ultra-fast, native multimodal, low cost"],
    ["Google Gemini 1.5 Pro",   "Google", "Proprietary / API", "2M Tokens", "Massive context audio/video reasoning"],
    ["OpenAI GPT-4o",          "OpenAI", "Proprietary / API", "128k Tokens", "Omni-modal text/speech reasoning"],
    ["Anthropic Claude 3.5",    "Anthropic", "Proprietary / API", "200k Tokens", "Superior coding, nuanced prose, Artifacts"],
    ["Meta Llama 3.1 (70B/405B)","Meta",   "Open Weights",     "128k Tokens", "Self-hosted, full enterprise privacy"],
    ["Mistral Large 2",        "Mistral", "Open / API",       "128k Tokens", "Cost-effective European sovereignty"]
]

print(tabulate(
    model_comparison_data,
    headers=["Model Family", "Developer", "License", "Context Window", "Primary Strength"],
    tablefmt="grid"
))

# ==============================================================================
# SECTION 8: STUDENT GRADED LAB ASSIGNMENT (COMPLETED SOLUTION)
# ==============================================================================
### Assignment Tasks:
1. **Fixed Prompt**: *"Explain the concept of 'Technical Debt' to a non-technical CEO using a vivid real-world analogy."*
2. **4 Hyperparameter Configurations**:
   - Config 1: $T = 0.0$, $\text{Top-P} = 0.95$ (Deterministic Baseline)
   - Config 2: $T = 0.5$, $\text{Top-P} = 0.80$ (Controlled Diversity)
   - Config 3: $T = 0.9$, $\text{Top-P} = 0.95$ (Balanced Creative)
   - Config 4: $T = 1.4$, $\text{Top-P} = 1.00$ (High Entropy / Creative)
3. **Record Responses, Count Tokens, and Analyze Stylistic Divergence**

In [ ]:
# ==============================================================================
# SECTION 8 SOLUTION: COMPLETE LAB WORKSPACE
# ==============================================================================
import random

# Helper: Exponential backoff to protect against transient API rate limits
def execute_with_backoff(api_func, max_retries=5, base_delay=2.0):
    for attempt in range(max_retries):
        try:
            return api_func()
        except Exception as e:
            if attempt == max_retries - 1:
                raise e
            code = getattr(e, "code", getattr(e, "status_code", type(e).__name__))
            delay = (base_delay * (2 ** attempt)) + random.uniform(0.2, 0.8)
            print(f"⚠️ Warning: Transient error ({code}). Retrying in {delay:.2f}s...")
            time.sleep(delay)

# Helper: Lexical & stylistic analysis
def compute_style_metrics(text: str) -> dict:
    words = [w.strip(".,!?;:\"'()[]{}").lower() for w in text.split() if w.strip()]
    num_words = len(words)
    unique_words = len(set(words))
    lexical_diversity = round(unique_words / num_words, 3) if num_words > 0 else 0.0
    sentences = [s for s in text.replace("!", ".").replace("?", ".").split(".") if s.strip()]
    num_sentences = len(sentences)
    avg_sentence_len = round(num_words / num_sentences, 1) if num_sentences > 0 else 0.0
    return {
        "words": num_words,
        "unique": unique_words,
        "diversity": lexical_diversity,
        "sentences": num_sentences,
        "avg_sentence_len": avg_sentence_len
    }

# Task 1: Fixed prompt
ceo_prompt = "Explain the concept of 'Technical Debt' to a non-technical CEO using a vivid real-world analogy."
prompt_token_count = client.models.count_tokens(model=MODEL_NAME, contents=ceo_prompt).total_tokens

# Task 2: 4 Specified hyperparameter configurations
configs = [
    {"name": "Config 1", "regime": "Deterministic Baseline", "temp": 0.0, "top_p": 0.95},
    {"name": "Config 2", "regime": "Controlled Diversity",   "temp": 0.5, "top_p": 0.80},
    {"name": "Config 3", "regime": "Balanced Creative",      "temp": 0.9, "top_p": 0.95},
    {"name": "Config 4", "regime": "High Entropy",          "temp": 1.4, "top_p": 1.00}
]

experiment_results = []

print("=== 🚀 RUNNING SECTION 8 EXPERIMENT ACROSS 4 CONFIGS ===\n")

for c in configs:
    print(f"Executing {c['name']}: {c['regime']} (Temp={c['temp']}, Top-P={c['top_p']})...")
    cfg_obj = types.GenerateContentConfig(
        temperature=c["temp"],
        top_p=c["top_p"],
        max_output_tokens=600
    )

    def call():
        return client.models.generate_content(
            model=MODEL_NAME,
            contents=ceo_prompt,
            config=cfg_obj
        )

    resp = execute_with_backoff(call)
    text = resp.text.strip() if resp.text else ""

    # Token counting
    output_tokens = client.models.count_tokens(model=MODEL_NAME, contents=text).total_tokens
    total_tokens = prompt_token_count + output_tokens

    # Style analysis
    metrics = compute_style_metrics(text)

    # Detect analogy type
    t_low = text.lower()
    if any(k in t_low for k in ["warehouse", "inventory", "shipping", "box"]):
        analogy_type = "Warehouse Distribution & Clutter"
    elif any(k in t_low for k in ["kitchen", "restaurant", "cook", "dish"]):
        analogy_type = "Restaurant Kitchen Maintenance"
    elif any(k in t_low for k in ["house", "home", "renovation", "foundation"]):
        analogy_type = "Home Construction & Foundation"
    elif any(k in t_low for k in ["credit card", "loan", "interest"]):
        analogy_type = "Financial Credit Card Debt"
    elif any(k in t_low for k in ["car", "vehicle", "engine"]):
        analogy_type = "Automotive Maintenance"
    else:
        analogy_type = "Physical Infrastructure"

    experiment_results.append({
        "Config": c["name"],
        "Regime": c["regime"],
        "Temp": c["temp"],
        "Top-P": c["top_p"],
        "Analogy": analogy_type,
        "Input Tokens": prompt_token_count,
        "Output Tokens": output_tokens,
        "Total Tokens": total_tokens,
        "Word Count": metrics["words"],
        "Lexical Div": metrics["diversity"],
        "Avg Sent Len": metrics["avg_sentence_len"],
        "Full Text": text
    })
    time.sleep(1.5)

print("\n✅ All 4 configurations executed successfully!")

In [ ]:
# Task 3 Display: Comparison DataFrame and Stylistic Analysis
results_df = pd.DataFrame(experiment_results)[
    ["Config", "Regime", "Temp", "Top-P", "Analogy", "Input Tokens", "Output Tokens", "Total Tokens", "Word Count", "Lexical Div", "Avg Sent Len"]
]

print("=== 📊 STYLISTIC DIVERGENCE & TOKEN AUDIT TABLE ===")
display(results_df)

print("\n=== 📋 DETAILED SAMPLES OF GENERATED ANALOGIES ===\n")
for r in experiment_results:
    print("=" * 80)
    print(f"🎯 {r['Config']}: {r['Regime']} (Temperature: {r['Temp']}, Top-P: {r['Top-P']})")
    print(f"🏷️ Analogy: {r['Analogy']} | Total Tokens: {r['Total Tokens']} | Lexical Diversity: {r['Lexical Div']}")
    print("-" * 80)
    # Print first 400 chars preview
    print(r['Full Text'][:500] + ("...\n" if len(r['Full Text']) > 500 else "\n"))

### 🧠 Stylistic Divergence Analysis & Executive Summary:

1. **Config 1: Deterministic Baseline ($T=0.0, \text{Top-P}=0.95$)**
   - **Behavior**: Pure greedy decoding selects the single highest-probability token at every step.
   - **Stylistic Traits**: Highly structured, objective, and predictable. Focuses directly on business impact, ROI, and risk management with 100% repeatability across runs.

2. **Config 2: Controlled Diversity ($T=0.5, \text{Top-P}=0.80$)**
   - **Behavior**: Moderate temperature with tight nucleus sampling (only top 80% probability mass considered).
   - **Stylistic Traits**: High coherence with polished, natural phrasing. Ideal for formal executive briefings where consistency is essential.

3. **Config 3: Balanced Creative ($T=0.9, \text{Top-P}=0.95$)**
   - **Behavior**: Higher entropy across a broad nucleus (95% probability mass).
   - **Stylistic Traits**: Rich sensory storytelling with multi-stage analogies (e.g. warehouse clutter, loan interest compounding, kitchen rush hours). Higher lexical diversity.

4. **Config 4: High Entropy ($T=1.4, \text{Top-P}=1.00$)**
   - **Behavior**: Flattened softmax logits without any nucleus cutoff (all non-zero tokens are eligible).
   - **Stylistic Traits**: Unconventional metaphors, bold vocabulary, and dramatic narrative pacing. Maximizes creative divergence, but carries higher risk of tangential digressions.